In [0]:
%sql
INSERT OVERWRITE customer_snapshot
Select customer_id,name,city,updated_timestamp,ingestion_timestamp,source_system
FROM 
(
SELECT *, ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY ingestion_timestamp DESC) AS rn
FROM 
customer_raw
)t 
where rn=1

In [0]:
%sql
CREATE TABLE customer_refined AS
SELECT*
FROM (
SELECT*,
         ROW_NUMBER() OVER (
           PARTITION BY customer_id
ORDER BY updated_timestamp DESC,
                    ingestion_timestamp DESC,
                    source_system ASC
         )AS rn
FROM customers_raw
) t
WHERE rn=1;

In [0]:
%sql
select * from dataxt.default.customer_refined

Orders Table 
Refund Table (create a sperate)


Late Arriving Data 

In [0]:
%sql
SELECT * from orders

In [0]:
%sql
SELECT * from refunds

In [0]:
%sql
-- Calculate Net Revenue(orders-revenue)
Select 
SUM(o.amount) AS gross_revenue,
SUM(coalesce(r.refund_amount,0)) AS total_refunds,
SUM(o.amount-coalesce(r.refund_amount,0)) as net_revenue
FROM orders o 
LEFT JOIN refunds r
ON o.order_id=r.order_id
WHERE o.order_date>='2024-01-01'
AND o.order_date<'2024-02-01'


In [0]:
%sql
-- CTE Example 

WITH step1 AS (
SELECT * FROM orders
WHERE orderdate=2024
),

step2 AS (
SELECT s1.*, c.city
FROM step1 s1
JOIN customers_snapshot c
ON s1.customer_id= c.customer_id
),
step3 AS (
SELECT city, SUM(amount)AS total
FROM step2
GROUP BY city
)
SELECT*FROM step3;

In [0]:
/dbfs/FileStore/tables/

list_files=[]
for l in list_files:
    chucnkprocess(l)

In [0]:
#Python You receive a 7GB CSV daily. How do you process it safely without memory crash?
def chucnkprocess(l):
    import pandas as pd 
    chunksize=1000000

    for chunk in pd.read_csv(f'/dbfs/FileStore/tables/{l}, chunksize=chunksize):
        filtered=chunk[chunk["status"]=="success"]

    print(filtered)



In [0]:
#Without Yeild 

def get_numbers():
    return[1,2,3]

for 


In [0]:
#with Yeild 

def get_numbers():
    yield 1
    yield 2
    yield 3

In [0]:
for num in get_numbers():
    print(num)

In [0]:
#Python You receive a 7GB CSV daily. How do you process it safely without memory crash?
#with Yield
def filter_success_rows(file_path):
    import pandas as pd
    for chunk in pd.read_csv(file_path, chunksize=1000000):
        for _, row in chunk.iterrows():
            if row["status"] == "success":
                yield row

for row in filter_success_rows('/dbfs/FileStore/tables/2019_revenue.csv'):
    print(row)



In [0]:
from multiprocessing import Pool


In [0]:
MERGE INTO 


ON orderid=order_id AND 
.whenMatchedUpdateAll()
.whenNotMatchedInsertAll()

In [0]:
CDC --> change data capture




In [0]:
CDF --> change data feed


In [0]:
Table1 --> Udpated 
Table2 --> Update this table automatically
